# 05 — County covariates (Ohio)

**Question:** What county characteristics, other than noise, could explain differences in CVD across Ohio counties?

**Data:**
- **Urbanicity:** 2020 Census blocks (`UR20` urban/rural flag, `POP20`) → % of residents in urban blocks
- **Demographics and socioeconomic status:** American Community Survey 5-year estimates, 2018–2022 (Census API)
- **Smoking:** CDC PLACES 2025 county data, age-adjusted current smoking (`CSMOKING`)
- **Air pollution (PM2.5):** to be added

**Selection principle:** include confounders that plausibly affect both noise exposure and CVD. Exclude variables on the causal pathway from noise to CVD (high blood pressure, short sleep). With 88 counties, the final model is limited to about 6–8 predictors, so the covariate set is fixed before modeling.

**Output:** `data/processed/covariates_oh_county.csv`: one row per county.

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import requests

RAW         = Path("../data/raw")
PROCESSED   = Path("../data/processed")
BLOCKS_PATH = Path("../data/large/census_blocks/tl_2022_39_tabblock20")   # same as notebook 03
OHIO_POP    = 11_799_448

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import os
from dotenv import load_dotenv

load_dotenv("../.env")   # the notebook runs from notebooks/, so .env is one folder up
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY")
assert CENSUS_API_KEY, "CENSUS_API_KEY not found. Check that ../.env exists and has the key."

RAW         = Path("../data/raw")
PROCESSED   = Path("../data/processed")
BLOCKS_PATH = Path("../data/large/census_blocks/tl_2022_39_tabblock20")   # same as notebook 03
OHIO_POP    = 11_799_448

In [2]:
blocks = gpd.read_file(BLOCKS_PATH, columns=["GEOID20", "POP20", "UR20"], ignore_geometry=True)
print(blocks["UR20"].value_counts(dropna=False))   # check the codes before using them

blocks["county_fips"] = blocks["GEOID20"].str[:5]
blocks["pop_urban"] = blocks["POP20"] * (blocks["UR20"] == "U")

urban = blocks.groupby("county_fips")[["POP20", "pop_urban"]].sum().reset_index()
urban["pct_urban"] = urban["pop_urban"] / urban["POP20"]

assert len(urban) == 88 and urban["POP20"].sum() == OHIO_POP
urban["pct_urban"].describe().round(3)

UR20
U    151568
R    124860
Name: count, dtype: int64


count    88.000
mean      0.467
std       0.281
min       0.000
25%       0.284
50%       0.446
75%       0.668
max       0.994
Name: pct_urban, dtype: float64

In [6]:
ACS_YEAR = 2022   # 2018–2022 5-year estimates

AGE65 = [f"B01001_{i:03d}E" for i in list(range(20, 26)) + list(range(44, 50))]   # men and women 65+
BACH  = [f"B15003_{i:03d}E" for i in range(22, 26)]                               # bachelor's, master's, professional, doctorate
VARS  = ["B01001_001E",                      # total population (age table)
         "B19013_001E",                      # median household income
         "B17001_001E", "B17001_002E",       # poverty universe; people below poverty
         "B03002_001E", "B03002_004E", "B03002_012E",   # total; non-Hispanic Black; Hispanic
         "B15003_001E"] + AGE65 + BACH       # adults 25+, then the age and degree columns

params = {
    "get": ",".join(["NAME"] + VARS),
    "for": "county:*",
    "in": "state:39",
    "key": CENSUS_API_KEY,
}

r = requests.get(f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5", params=params, timeout=60)
r.raise_for_status()
if "json" not in r.headers.get("Content-Type", ""):
    raise RuntimeError(f"Census API did not return JSON (status {r.status_code}):\n{r.text[:500]}")
data = r.json()

acs_raw = pd.DataFrame(data[1:], columns=data[0])
acs_raw.to_csv(RAW / "acs5_2022_county_OH.csv", index=False)   # raw copy, as with PLACES
acs_raw.head()

,NAME,B01001_001E,B19013_001E,B17001_001E,B17001_002E,B03002_001E,B03002_004E,B03002_012E,B15003_001E,B01001_020E,...,B01001_046E,B01001_047E,B01001_048E,B01001_049E,B15003_022E,B15003_023E,B15003_024E,B15003_025E,state,county
0,"Adams County, Ohio",27509,46234,27050,5313,27509,97,303,18788,539,...,589,629,364,162,1514,1003,246,97,39,001
1,"Allen County, Ohio",102087,58976,98017,12081,102087,10914,3548,68467,1237,...,2702,2123,1139,1516,8167,3649,748,544,39,003
2,"Ashland County, Ohio",52522,62254,50670,5950,52522,408,883,35796,649,...,1215,1070,682,838,5103,2210,418,109,39,005
3,"Ashtabula County, Ohio",97666,53663,94515,17024,97666,3385,4625,68664,1353,...,2711,1917,1203,1325,7102,2875,523,265,39,007
4,"Athens County, Ohio",61276,48750,53380,13049,61276,1507,1231,36121,586,...,1384,862,391,744,5632,4907,443,1397,39,009


In [7]:
acs = acs_raw.copy()
acs["county_fips"] = acs["state"] + acs["county"]
num = acs[VARS].apply(pd.to_numeric)

acs_cov = pd.DataFrame({
    "county_fips":        acs["county_fips"],
    "pct_65plus":         num[AGE65].sum(axis=1) / num["B01001_001E"],
    "pct_poverty":        num["B17001_002E"] / num["B17001_001E"],
    "median_hh_income":   num["B19013_001E"],
    "pct_bachelors_plus": num[BACH].sum(axis=1) / num["B15003_001E"],
    "pct_black_nh":       num["B03002_004E"] / num["B03002_001E"],
    "pct_hispanic":       num["B03002_012E"] / num["B03002_001E"],
})

assert len(acs_cov) == 88
assert (acs_cov["median_hh_income"] > 0).all(), "negative values are Census missing-data codes"
pct = [c for c in acs_cov.columns if c.startswith("pct_")]
assert acs_cov[pct].ge(0).all().all() and acs_cov[pct].le(1).all().all()
acs_cov.describe().round(3)

,pct_65plus,pct_poverty,median_hh_income,pct_bachelors_plus,pct_black_nh,pct_hispanic
count,88.000,88.000,88.000,88.000,88.000,88.000
mean,0.187,0.130,65302.102,0.223,0.040,0.031
std,0.025,0.042,13171.972,0.090,0.056,0.024
min,0.125,0.045,46234.000,0.107,0.001,0.002
25%,0.177,0.101,56380.750,0.158,0.009,0.014
50%,0.186,0.129,63014.500,0.190,0.019,0.021
75%,0.198,0.161,70670.250,0.270,0.045,0.038
max,0.284,0.244,123995.000,0.574,0.287,0.107


In [8]:
places_raw = pd.read_csv(RAW / "places_2025_county_OH.csv", dtype={"locationid": str})
smk = places_raw[(places_raw["measureid"] == "CSMOKING") & (places_raw["datavaluetypeid"] == "AgeAdjPrv")]
smoking = smk[["locationid", "data_value"]].rename(
    columns={"locationid": "county_fips", "data_value": "pct_smoking"}
)
smoking["pct_smoking"] = smoking["pct_smoking"] / 100     # PLACES reports percent; store as a 0–1 share like the others
assert len(smoking) == 88

In [9]:
cov = (urban[["county_fips", "pct_urban"]]
       .merge(acs_cov, on="county_fips", how="outer", validate="one_to_one")
       .merge(smoking, on="county_fips", how="outer", validate="one_to_one"))
assert len(cov) == 88 and cov.notna().all().all()

cov.drop(columns="county_fips").corr().round(2)

,pct_urban,pct_65plus,pct_poverty,median_hh_income,pct_bachelors_plus,pct_black_nh,pct_hispanic,pct_smoking
pct_urban,1.00,-0.29,-0.09,0.26,0.66,0.67,0.39,-0.56
pct_65plus,-0.29,1.00,0.13,-0.42,-0.43,-0.17,-0.08,0.28
pct_poverty,-0.09,0.13,1.00,-0.82,-0.39,0.18,-0.22,0.72
median_hh_income,0.26,-0.42,-0.82,1.00,0.73,-0.01,0.16,-0.82
pct_bachelors_plus,0.66,-0.43,-0.39,0.73,1.00,0.45,0.25,-0.85
pct_black_nh,0.67,-0.17,0.18,-0.01,0.45,1.00,0.29,-0.30
pct_hispanic,0.39,-0.08,-0.22,0.16,0.25,0.29,1.00,-0.44
pct_smoking,-0.56,0.28,0.72,-0.82,-0.85,-0.30,-0.44,1.00


In [10]:
cov.to_csv(PROCESSED / "covariates_oh_county.csv", index=False)

In [11]:
cov["pct_smoking"].describe()

count    88.000000
mean      0.175000
std       0.026168
min       0.099000
25%       0.156000
50%       0.176000
75%       0.194000
max       0.233000
Name: pct_smoking, dtype: float64

## Interpretation

### Findings
- All 88 counties have complete covariate data (no missing values; no Census missing-data codes).
- Covariate ranges across Ohio counties:

| Covariate | Median | Range |
|---|---|---|
| % urban (2020 Census blocks) | 44.6% | 0.0% – 99.4% |
| % age 65+ | 18.6% | 12.5% – 28.4% |
| % below poverty | 12.9% | 4.5% – 24.4% |
| Median household income | $63,015 | $46,234 – $123,995 |
| % bachelor's degree or higher | 19.0% | 10.7% – 57.4% |
| % non-Hispanic Black | 1.9% | 0.1% – 28.7% |
| % Hispanic | 2.1% | 0.2% – 10.7% |
| Current smoking (age-adjusted) | 17.6% | 9.9% – 23.3% |

### Correlations between covariates
- The three SES measures are highly intercorrelated (poverty vs. income r = −0.82; income vs. bachelor's+ r = 0.73), so only one can be included.
- **Poverty** is the least correlated with % urban (r = −0.09, vs. 0.26 for income and 0.66 for bachelor's+) and with smoking (r = 0.72, vs. −0.82 and −0.85), so it adds the most distinct information.
- **Smoking is strongly tied to SES** (r = −0.85 with bachelor's+). This partly reflects a real SES gradient and partly that PLACES smoking estimates are model-based and derived in part from demographics.
- **% non-Hispanic Black is correlated with % urban** (r = 0.67), reflecting Ohio's concentration of Black residents in urban counties.

### Decisions
- **Primary model covariates:** % urban, % 65+, % poverty, % non-Hispanic Black, smoking, and PM2.5 (to be added). With noise, that's 7 predictors for 88 counties.
- **Excluded as mediators** (on the causal path from noise to CVD): high blood pressure, short sleep.
- **Kept in the file for sensitivity analyses only:** median income, % bachelor's+, % Hispanic.
- **To check in notebook 06:** variance inflation factors for smoking and % non-Hispanic Black. If smoking's VIF exceeds 5, report models with and without it.

### Output
`data/processed/covariates_oh_county.csv`: one row per county (PM2.5 to be added).